# ทำความสะอาดข้อมูล ขั้นที่ 4/5: มาตรฐานรูปแบบ + ความสอดคล้องของตัวเลข

| ขั้น | เรื่อง | สถานะ |
|---|---|---|
| 1 | โครงสร้าง คีย์หลัก ข้อมูลซ้ำ ขอบเขต | ✅ |
| 2 | ค่าว่าง | ✅ |
| 3 | ค่าผิดปกติ | ✅ |
| **4 (ไฟล์นี้)** | **มาตรฐานรูปแบบ + ความสอดคล้อง + ความเป็นปัจจุบัน** | เกณฑ์ 3.5.1.1.3, 3.5.1.2.3, 3.5.1.2.4 |
| 5 | Data Dictionary + ไฟล์สะอาดสุดท้าย | |

ขั้นนี้ทำ 4 เรื่อง
1. **หน่วย (Units)**: ยืนยันว่าทุกคอลัมน์ใช้หน่วยเดียวกันทั้งไฟล์
2. **รูปแบบ (Format)**: ชนิดข้อมูล ทศนิยม การจัดหมวดหมู่ (เพิ่มคอลัมน์ทวีป/อาเซียน ให้แดชบอร์ดกรองได้)
3. **ความสอดคล้องของตัวเลข (Consistency)**: ส่วนย่อยรวมได้เท่าทั้งหมด, สัดส่วนรวมได้ 100% และกระทบยอด (reconciliation) กับข้อมูลก่อนทำความสะอาด
4. **ความเป็นปัจจุบัน (Timeliness)**: ติดป้ายปีที่ข้อมูลยังไม่ครบ

**อินพุต:** `clean_output/clean_step3.csv` → **เอาต์พุต:** `clean_step4.csv`, `consistency_report.csv`, `reconciliation.csv` และเพิ่มบรรทัดใน `cleaning_log.csv`

> รัน **Kernel → Restart Kernel and Run All Cells** (ต้องรันขั้นที่ 3 สำเร็จก่อน)

## 4.1 โหลดผลจากขั้นที่ 3

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

OUT_DIR = "clean_output"
IN_FILE = os.path.join(OUT_DIR, "clean_step3.csv")
LOG_FILE = os.path.join(OUT_DIR, "cleaning_log.csv")
if not os.path.exists(IN_FILE):
    raise FileNotFoundError("ไม่พบ clean_output/clean_step3.csv ให้รัน clean_step3_outliers.ipynb ให้สำเร็จก่อน")

df = pd.read_csv(IN_FILE).sort_values(["country", "year"]).reset_index(drop=True)

log_old = pd.read_csv(LOG_FILE)
log_old = log_old[~log_old["step"].str.startswith("4-")]
log_rows = []
def log_step(action, before, after, reason):
    log_rows.append({"step": "4-standardize", "action": action,
                     "rows_before": before.shape[0], "rows_after": after.shape[0],
                     "cols_before": before.shape[1], "cols_after": after.shape[1], "reason": reason})
    print(f"✔ {action}: แถว {before.shape[0]:,} → {after.shape[0]:,} | คอลัมน์ {before.shape[1]} → {after.shape[1]}")

print("ข้อมูลจากขั้นที่ 3:", df.shape, "| ประเทศ:", df["country"].nunique())

## 4.2 หน่วย (Units): ตรวจและยืนยัน
แหล่งข้อมูล (Our World in Data) ตั้งชื่อคอลัมน์ตามหน่วยอยู่แล้ว เราจึงกำหนดหน่วยจาก **ส่วนท้ายของชื่อคอลัมน์** แล้วตรวจว่าทุกคอลัมน์มีหน่วยครบ ไม่มีคอลัมน์ไหนตกหล่น

| ส่วนท้ายชื่อ | หน่วย | ความหมาย |
|---|---|---|
| `_electricity`, `electricity_generation/demand`, `net_elec_imports`, `primary_energy_consumption` | **TWh** | เทราวัตต์-ชั่วโมง = 1,000 ล้าน kWh |
| `_share_elec` | **%** | ร้อยละของไฟฟ้าที่ผลิตทั้งหมด |
| `per_capita_electricity`, `energy_per_capita` | **kWh/คน** | ต่อประชากร 1 คน ต่อปี |
| `carbon_intensity_elec` | **gCO₂/kWh** | กรัมคาร์บอนไดออกไซด์ต่อไฟฟ้า 1 หน่วย |
| `greenhouse_gas_emissions` | **MtCO₂e** | ล้านตันคาร์บอนเทียบเท่า จากภาคผลิตไฟฟ้า |
| `gdp` | **international-$** | ดอลลาร์ปรับค่าครองชีพ (ราคาคงที่ปี 2011) |
| `population` | **คน** | |

In [ ]:
def unit_of(col):
    if col in ["country", "iso_code", "region"]: return "text"
    if col == "year": return "year (ค.ศ.)"
    if col.startswith("is_") or col in ["gdp_available", "low_precision", "year_complete"]: return "boolean"
    if "share_elec" in col: return "%"
    if col in ["per_capita_electricity", "energy_per_capita"]: return "kWh/person"
    if col == "carbon_intensity_elec": return "gCO2/kWh"
    if col == "greenhouse_gas_emissions": return "MtCO2e"
    if col == "gdp": return "international-$ (2011 prices)"
    if col == "population": return "persons"
    if col.endswith("_electricity") or col in ["electricity_generation", "electricity_demand",
                                               "net_elec_imports", "primary_energy_consumption"]: return "TWh"
    return None

units = pd.Series({c: unit_of(c) for c in df.columns}, name="unit")
print("คอลัมน์ที่ยังไม่มีหน่วย:", units[units.isna()].index.tolist() or "ไม่มี ✅")
units.value_counts().to_frame("จำนวนคอลัมน์")

In [ ]:
# ตรวจว่าหน่วยสอดคล้องกันจริง: แปลงหน่วยแล้วตัวเลขต้องตรงกัน
# 1 TWh = 1,000,000,000 kWh → ไฟฟ้าต่อคน (kWh) = TWh x 1e9 / ประชากร
calc = df["electricity_generation"] * 1e9 / df["population"]
ratio = (calc / df["per_capita_electricity"]).dropna()
print(f"TWh → kWh/คน: อัตราส่วนคำนวณ/ข้อมูล อยู่ระหว่าง {ratio.min():.4f} ถึง {ratio.max():.4f} (ควรเป็น 1)")

# ช่วงค่าของแต่ละหน่วย (ตัวอย่างค่าหน่วย/โดเมน ตามเกณฑ์ 3.5.1.1)
domain = pd.DataFrame({"unit": units, "min": df.min(numeric_only=True), "median": df.median(numeric_only=True),
                       "max": df.max(numeric_only=True)}).dropna(subset=["min"])
domain[domain["unit"].isin(["TWh", "%", "kWh/person", "gCO2/kWh"])].round(2).head(15)

## 4.3 รูปแบบ (Format)
### ทศนิยม: กำหนดให้เท่ากันทั้งคอลัมน์ประเภทเดียวกัน
ข้อมูลดิบมีทศนิยมไม่เท่ากัน (บางค่า 3 ตำแหน่ง บางค่า 6 ตำแหน่ง) เวลาแสดงบนแดชบอร์ดจะดูไม่สม่ำเสมอ และเกณฑ์ 3.5.3.4.2 ต้องการ "การปัดทศนิยมสอดคล้องทั้งรายงาน"

| หน่วย | ทศนิยม | เหตุผล |
|---|---|---|
| TWh | 3 | ประเทศเล็กผลิตไฟแค่ 0.01 TWh ถ้าปัดน้อยกว่านี้จะกลายเป็น 0 |
| % | 2 | อ่านง่าย และละเอียดพอเห็นพลังงานแสงอาทิตย์ที่ยังน้อย |
| kWh/คน, gCO₂/kWh | 1 | ค่าเป็นหลักร้อยหลักพัน |
| คน, international-$ | 0 | เป็นจำนวนเต็ม |

In [ ]:
DECIMALS = {"TWh": 3, "%": 2, "kWh/person": 1, "gCO2/kWh": 1, "MtCO2e": 3,
            "persons": 0, "international-$ (2011 prices)": 0}

before = df.copy()
for col, u in units.items():
    if u in DECIMALS:
        df[col] = df[col].round(DECIMALS[u])
df["population"] = df["population"].astype("Int64")    # จำนวนเต็มที่รองรับค่าว่าง
df["gdp"] = df["gdp"].astype("Int64")
log_step("ปัดทศนิยมตามหน่วย + population/gdp เป็นจำนวนเต็ม", before, df, "ทศนิยมสม่ำเสมอทั้งคอลัมน์ประเภทเดียวกัน")

# ผลกระทบของการปัด: ต้องเล็กมากจนไม่มีนัยสำคัญ
num = [c for c in before.select_dtypes("number").columns if c in df.columns]
diff = (df[num].astype(float) - before[num]).abs().max().sort_values(ascending=False)
print("\nค่าที่เปลี่ยนมากที่สุดจากการปัด (ต่อคอลัมน์):")
diff.head(6).round(4)

### จัดหมวดหมู่: เพิ่มคอลัมน์ทวีปและอาเซียน
เกณฑ์ข้อ 3.5.2.2.3 ต้องการ **เปรียบเทียบข้ามกลุ่ม** และแดชบอร์ดต้องมี **ตัวกรอง** จึงเพิ่ม
- `region`: ทวีป (ใช้รหัส ISO ของประเทศจับคู่กับทวีปตามมาตรฐาน UN)
- `is_asean`: เป็นสมาชิกอาเซียนหรือไม่ (ตรงกับคำถามของเราที่เทียบไทยกับอาเซียน)

In [ ]:
REGION_CODES = {
    "Africa": ["AGO", "BDI", "BEN", "BFA", "BWA", "CAF", "CIV", "CMR", "COD", "COG", "COM", "CPV", "DJI", "DZA", "EGY", "ERI", "ESH", "ETH", "GAB", "GHA", "GIN", "GMB", "GNB", "GNQ", "KEN", "LBR", "LBY", "LSO", "MAR", "MDG", "MLI", "MOZ", "MRT", "MUS", "MWI", "NAM", "NER", "NGA", "REU", "RWA", "SDN", "SEN", "SLE", "SOM", "SSD", "STP", "SWZ", "SYC", "TCD", "TGO", "TUN", "TZA", "UGA", "ZAF", "ZMB", "ZWE"],
    "Asia": ["AFG", "ARE", "ARM", "AZE", "BGD", "BHR", "BRN", "BTN", "CHN", "CYP", "GEO", "HKG", "IDN", "IND", "IRN", "IRQ", "ISR", "JOR", "JPN", "KAZ", "KGZ", "KHM", "KOR", "KWT", "LAO", "LBN", "LKA", "MAC", "MDV", "MMR", "MNG", "MYS", "NPL", "OMN", "PAK", "PHL", "PRK", "PSE", "QAT", "SAU", "SGP", "SYR", "THA", "TJK", "TKM", "TLS", "TUR", "TWN", "UZB", "VNM", "YEM"],
    "Europe": ["ALB", "AUT", "BEL", "BGR", "BIH", "BLR", "CHE", "CZE", "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "FRO", "GBR", "GRC", "HRV", "HUN", "IRL", "ISL", "ITA", "LTU", "LUX", "LVA", "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL", "PRT", "ROU", "RUS", "SRB", "SVK", "SVN", "SWE", "UKR", "XKX"],
    "North America": ["ABW", "ATG", "BHS", "BLZ", "BRB", "CAN", "CRI", "CUB", "CYM", "DMA", "DOM", "GLP", "GRD", "GRL", "GTM", "HND", "HTI", "JAM", "KNA", "LCA", "MEX", "MSR", "MTQ", "NIC", "PAN", "PRI", "SLV", "SPM", "TCA", "TTO", "USA", "VCT", "VGB", "VIR"],
    "Oceania": ["ASM", "AUS", "COK", "FJI", "GUM", "KIR", "NCL", "NRU", "NZL", "PNG", "PYF", "SLB", "TON", "VUT", "WSM"],
    "South America": ["ARG", "BOL", "BRA", "CHL", "COL", "ECU", "FLK", "GUF", "GUY", "PER", "PRY", "SUR", "URY", "VEN"],
}

ISO_TO_REGION = {iso: region for region, codes in REGION_CODES.items() for iso in codes}
ASEAN_ISO = ["BRN", "KHM", "IDN", "LAO", "MYS", "MMR", "PHL", "SGP", "THA", "VNM"]

before = df.copy()
df["region"] = df["iso_code"].map(ISO_TO_REGION)
df["is_asean"] = df["iso_code"].isin(ASEAN_ISO)

# ตรวจ: ทุกประเทศต้องได้ทวีป (FK match rate ต้อง 100%)
no_region = df.loc[df["region"].isna(), "country"].unique()
print("ประเทศที่ไม่มีทวีป:", list(no_region) or "ไม่มี ✅")
print("อาเซียน:", sorted(df.loc[df["is_asean"], "country"].unique()), "→", df.loc[df["is_asean"], "country"].nunique(), "ประเทศ")
log_step("เพิ่มคอลัมน์ region และ is_asean", before, df, "ใช้กรอง/เทียบกลุ่มในแดชบอร์ด")

df.drop_duplicates("country").groupby("region").size().to_frame("จำนวนประเทศ")

In [ ]:
# ชนิดข้อมูลสุดท้าย: หมวดหมู่เป็น category (กรองเร็ว ใช้หน่วยความจำน้อย = ช่วยเรื่อง Performance)
for c in ["country", "iso_code", "region"]:
    df[c] = df[c].astype("category")
df["year"] = df["year"].astype(int)
df.dtypes.astype(str).value_counts().to_frame("จำนวนคอลัมน์")

## 4.4 ความสอดคล้องของตัวเลข (Consistency)
ข้อมูลที่ "สะอาด" ต้อง **ไม่ขัดแย้งกันเอง** ตรวจ 6 ข้อ โดยยอมให้คลาดเคลื่อนเล็กน้อยจากการปัดทศนิยม (tolerance)

| # | สิ่งที่ต้องเป็นจริง | ยอมคลาดเคลื่อน |
|---|---|---|
| 1 | ฟอสซิล + คาร์บอนต่ำ = ไฟฟ้าทั้งหมด (TWh) | 0.01 TWh |
| 2 | สัดส่วนฟอสซิล + สัดส่วนคาร์บอนต่ำ = 100% | 0.05 จุด % |
| 3 | ถ่านหิน + ก๊าซ + น้ำมัน = ฟอสซิล | 0.01 TWh |
| 4 | หมุนเวียน + นิวเคลียร์ = คาร์บอนต่ำ | 0.01 TWh |
| 5 | ความต้องการ = ผลิต + นำเข้าสุทธิ | 0.01 TWh |
| 6 | สัดส่วน (%) = ปริมาณ ÷ ไฟฟ้าทั้งหมด × 100 (ทุกเชื้อเพลิง) | 0.05 จุด % |

In [ ]:
gen = df["electricity_generation"]
SHARE_MAP = {
    "coal_share_elec": "coal_electricity", "gas_share_elec": "gas_electricity",
    "oil_share_elec": "oil_electricity", "nuclear_share_elec": "nuclear_electricity",
    "hydro_share_elec": "hydro_electricity", "solar_share_elec": "solar_electricity",
    "wind_share_elec": "wind_electricity", "biofuel_share_elec": "biofuel_electricity",
    "other_renewables_share_elec_exc_biofuel": "other_renewable_exc_biofuel_electricity",
    "fossil_share_elec": "fossil_electricity", "renewables_share_elec": "renewables_electricity",
    "low_carbon_share_elec": "low_carbon_electricity",
}

def run_checks(data):
    g = data["electricity_generation"]
    checks = {
        "1 ฟอสซิล+คาร์บอนต่ำ = ไฟฟ้าทั้งหมด": (data["fossil_electricity"] + data["low_carbon_electricity"] - g, 0.01),
        "2 สัดส่วนฟอสซิล+คาร์บอนต่ำ = 100%": (data["fossil_share_elec"] + data["low_carbon_share_elec"] - 100, 0.05),
        "3 ถ่านหิน+ก๊าซ+น้ำมัน = ฟอสซิล": (data[["coal_electricity", "gas_electricity", "oil_electricity"]].sum(axis=1, min_count=3)
                                       - data["fossil_electricity"], 0.01),
        "4 หมุนเวียน+นิวเคลียร์ = คาร์บอนต่ำ": (data["renewables_electricity"] + data["nuclear_electricity"]
                                           - data["low_carbon_electricity"], 0.01),
        "5 ความต้องการ = ผลิต+นำเข้าสุทธิ": (g + data["net_elec_imports"] - data["electricity_demand"], 0.01),
    }
    for share, amount in SHARE_MAP.items():
        checks[f"6 {share}"] = ((data[amount] / g * 100).where(g > 0) - data[share], 0.05)

    rows = []
    for name, (err, tol) in checks.items():
        err = err.abs().dropna()
        rows.append({"การตรวจ": name, "แถวที่ตรวจได้": len(err), "คลาดเคลื่อนสูงสุด": round(err.max(), 4),
                     "เกณฑ์": tol, "แถวที่เกินเกณฑ์": int((err > tol).sum())})
    out = pd.DataFrame(rows)
    out["ผล"] = np.where(out["แถวที่เกินเกณฑ์"] == 0, "✅", "❌")
    return out

consistency = run_checks(df.astype({"population": float}))
consistency

In [ ]:
# ถ้ามีข้อใดไม่ผ่าน: ยึด "ปริมาณ (TWh)" เป็นค่าหลัก แล้วคำนวณสัดส่วนใหม่ (single source of truth)
n_bad = consistency.loc[consistency["การตรวจ"].str.startswith("6"), "แถวที่เกินเกณฑ์"].sum()
if n_bad > 0:
    before = df.copy()
    for share, amount in SHARE_MAP.items():
        df[share] = (df[amount] / gen * 100).where(gen > 0).round(2)
    log_step(f"คำนวณสัดส่วนใหม่จากปริมาณ ({n_bad} แถวที่ไม่สอดคล้อง)", before, df, "ยึดปริมาณ TWh เป็นค่าหลัก")
    consistency = run_checks(df.astype({"population": float}))
    display(consistency)
else:
    print("ทุกการตรวจผ่าน ✅ ไม่ต้องแก้ไข (หลังปัดทศนิยมแล้วก็ยังสอดคล้องกัน)")

### กระทบยอด (Reconciliation): ตัวเลขรวมหลังทำความสะอาด "หายไป" เพราะอะไร
เกณฑ์ 3.5.1.2.4 ต้องการ **reconciliation note** ว่ายอดรวม จำนวน และค่าเฉลี่ยหลังทำความสะอาด ไม่ได้ผิดเพี้ยนโดยไม่มีเหตุผล
เราเทียบ **ไฟฟ้าที่ผลิตทั้งโลกต่อปี** ระหว่างข้อมูลหลังขั้นที่ 1 (ยังไม่แก้ค่าใด ๆ) กับข้อมูลหลังขั้นที่ 4

In [ ]:
s1 = pd.read_csv(os.path.join(OUT_DIR, "clean_step1.csv"))
recon = pd.DataFrame({
    "ประเทศ_ขั้น1": s1.groupby("year")["country"].nunique(),
    "ประเทศ_ขั้น4": df.groupby("year", observed=True)["country"].nunique(),
    "ไฟฟ้าโลก_ขั้น1_TWh": s1.groupby("year")["electricity_generation"].sum().round(1),
    "ไฟฟ้าโลก_ขั้น4_TWh": df.groupby("year")["electricity_generation"].sum().round(1),
})
recon["ต่าง_TWh"] = (recon["ไฟฟ้าโลก_ขั้น4_TWh"] - recon["ไฟฟ้าโลก_ขั้น1_TWh"]).round(1)
recon["ต่าง_%"] = (recon["ต่าง_TWh"] / recon["ไฟฟ้าโลก_ขั้น1_TWh"] * 100).round(3)
recon.loc[[2000, 2010, 2020, 2021, 2022]]

In [ ]:
# อธิบายส่วนต่างทุกปีได้หรือไม่
kosovo_2022 = s1.loc[(s1["country"] == "Kosovo") & (s1["year"] == 2022), "electricity_generation"].sum()
print("ส่วนต่างปี 2000-2021 สูงสุด:", recon.loc[:2021, "ต่าง_TWh"].abs().max(), "TWh (มาจากการปัดทศนิยมเท่านั้น)")
print("ส่วนต่างปี 2022:", recon.loc[2022, "ต่าง_TWh"], "TWh | ไฟฟ้าโคโซโว 2022 ที่ตั้งเป็นค่าว่างในขั้นที่ 3:", kosovo_2022, "TWh")

> **Reconciliation note (คัดลอกไปใส่สไลด์ได้):**
> - ยอดไฟฟ้าโลกปี 2000–2021 หลังทำความสะอาด **เท่ากับก่อนทำความสะอาด** (ต่างแค่การปัดทศนิยม) เพราะ 0 ปลอมที่ลบในขั้นที่ 2 มีค่าเป็น 0 อยู่แล้ว และเขตปกครองที่ลบไม่มีข้อมูลไฟฟ้า จึงไม่กระทบยอดรวม
> - จำนวนประเทศลดลง 9 ประเทศ ตามที่ลบในขั้นที่ 2 (ไม่มีข้อมูลไฟฟ้าเลย)
> - ปี 2022 ยอดลดลงเท่ากับไฟฟ้าโคโซโวที่ตั้งเป็นค่าว่างในขั้นที่ 3 พอดี
>
> **สรุป:** ทุกส่วนต่างอธิบายได้ ไม่มีตัวเลขหายไปโดยไม่มีเหตุผล

## 4.5 ความเป็นปัจจุบัน (Timeliness): ติดป้ายปีที่ข้อมูลไม่ครบ
ปี 2022 มีข้อมูลไม่ถึงครึ่งของประเทศ ถ้าแดชบอร์ดแสดง "ยอดรวมโลกปี 2022" จะดูเหมือนไฟฟ้าโลกลดลงฮวบ ซึ่ง **ผิด** (จริง ๆ คือข้อมูลยังมาไม่ครบ)

In [ ]:
COMPLETE_THRESHOLD = 90   # ปีที่มีข้อมูลไฟฟ้า >= 90% ของประเทศ ถือว่าครบ
coverage = df.groupby("year")["electricity_generation"].count() / df["country"].nunique() * 100

before = df.copy()
df["year_complete"] = df["year"].map(coverage >= COMPLETE_THRESHOLD)
LATEST_COMPLETE = int(coverage[coverage >= COMPLETE_THRESHOLD].index.max())
log_step("เพิ่มคอลัมน์ year_complete", before, df,
         f"ปีที่มีข้อมูลไฟฟ้า < {COMPLETE_THRESHOLD}% ของประเทศ = ข้อมูลเบื้องต้น")

ax = coverage.plot.bar(figsize=(10, 3), color=["tab:red" if v < COMPLETE_THRESHOLD else "tab:blue" for v in coverage])
ax.axhline(COMPLETE_THRESHOLD, color="grey", ls="--"); ax.set_ylabel("% countries with data")
ax.set_title("Data completeness by year (red = incomplete)"); plt.show()

print("ปีที่ข้อมูลไม่ครบ:", coverage[coverage < COMPLETE_THRESHOLD].index.tolist())
print("ปีล่าสุดที่ข้อมูลครบ (ใช้เป็นค่าเริ่มต้นของแดชบอร์ด):", LATEST_COMPLETE)

In [ ]:
# ข้อมูลประเทศไทยปี 2022 มีครบไหม (ถ้ามี แดชบอร์ดรายประเทศยังแสดงปี 2022 ได้)
th = df[df["country"] == "Thailand"].set_index("year")
has22 = bool(2022 in th.index and pd.notna(th.loc[2022, "electricity_generation"]))
print("ไทยมีข้อมูลไฟฟ้าปี 2022:", "มี ✅" if has22 else "ไม่มี")
print("อาเซียนที่มีข้อมูลปี 2022:", sorted(df[(df["is_asean"]) & (df["year"] == 2022) & df["electricity_generation"].notna()]["country"]))

> **กติกาสำหรับแดชบอร์ด (จดไว้ใช้ตอนสร้าง):**
> - ภาพรวมโลก/ทวีป/ค่ามัธยฐาน ใช้ปีที่ `year_complete == True` เท่านั้น (ถึงปี 2021)
> - หน้ารายประเทศแสดงปี 2022 ได้ แต่ต้องมีหมายเหตุ "ข้อมูลเบื้องต้น"
> - ระบุบนแดชบอร์ด: **"ข้อมูล: Our World in Data, ปี 2000–2022 (ปี 2022 ข้อมูลเบื้องต้น)"**

## 4.6 จัดลำดับคอลัมน์ให้อ่านง่าย และบันทึก

In [ ]:
ID = ["country", "iso_code", "region", "is_asean", "year"]
FLAGS = ["year_complete", "gdp_available", "is_outlier_pc_elec", "low_precision"]
CONTEXT = ["population", "gdp"]
ELEC_MAIN = ["electricity_generation", "electricity_demand", "net_elec_imports", "per_capita_electricity",
             "carbon_intensity_elec", "greenhouse_gas_emissions"]
ENERGY = ["primary_energy_consumption", "energy_per_capita"]
TWH = [c for c in df.columns if c.endswith("_electricity") and c != "per_capita_electricity"]
SHARE = [c for c in df.columns if "share_elec" in c]

ORDER = ID + FLAGS + CONTEXT + ELEC_MAIN + ENERGY + TWH + SHARE
missing_in_order = set(df.columns) - set(ORDER)
assert not missing_in_order, f"ลืมคอลัมน์: {missing_in_order}"
df = df[ORDER]
print("คอลัมน์ทั้งหมด:", len(ORDER))
df[df["country"] == "Thailand"].tail(3)

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "clean_step4.csv"), index=False, encoding="utf-8-sig")
consistency.to_csv(os.path.join(OUT_DIR, "consistency_report.csv"), index=False, encoding="utf-8-sig")
recon.to_csv(os.path.join(OUT_DIR, "reconciliation.csv"), encoding="utf-8-sig")
pd.concat([log_old, pd.DataFrame(log_rows)]).to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("ข้อมูลหลังขั้นที่ 4:", df.shape, "| ประเทศ:", df["country"].nunique())
print("\nบันทึกแล้ว:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUT_DIR, f))/1024:,.0f} KB)")

In [ ]:
pd.read_csv(LOG_FILE)

## ✅ เช็กลิสต์ก่อนไปขั้นที่ 5
- [ ] รันครบไม่มี error และตารางความสอดคล้องข้อ 4.4 ขึ้น ✅ ทุกแถว
- [ ] ทุกประเทศมีทวีป (ข้อ 4.3) และอาเซียนครบ 10 ประเทศ
- [ ] มีไฟล์ใหม่ `clean_step4.csv`, `consistency_report.csv`, `reconciliation.csv`
- [ ] อธิบายได้ว่า **ทำไมต้องติดป้ายปี 2022** และ **ยอดรวมหลังทำความสะอาดต่างจากเดิมเพราะอะไร**

**ประโยคสรุปไว้ตอบกรรมการ:** "เรายืนยันหน่วยทุกคอลัมน์ ปัดทศนิยมให้สม่ำเสมอตามหน่วย เพิ่มทวีปและอาเซียนไว้กรอง ตรวจความสอดคล้อง 6 ข้อผ่านทั้งหมด กระทบยอดไฟฟ้าโลกกับข้อมูลก่อนทำความสะอาดแล้วอธิบายส่วนต่างได้ทุกปี และติดป้ายปี 2022 เป็นข้อมูลเบื้องต้น เพื่อไม่ให้แดชบอร์ดแสดงผลเข้าใจผิด"

**ขั้นถัดไป (ไฟล์ที่ 5 ขั้นสุดท้าย):** สร้าง Data Dictionary (ชื่อ คำอธิบายภาษาไทย ชนิด หน่วย ตัวอย่างค่า) ส่งออกไฟล์สะอาดสุดท้ายเป็น CSV + Excel + ไฟล์ตัวอย่างเล็ก และสรุป pipeline ทั้ง 5 ขั้นให้ทำซ้ำได้ แล้วจึงเริ่มสร้างแดชบอร์ด